# Shakespeare — Workshop Notebook
**CompLit 126x — Love in Context**

This notebook implements the prompt chain designed by the Shakespeare workshop group. Their chain used a **directional routing + quantified scoring + exhaustive research** architecture — the most elaborate chain of any group. Instead of iterating on a single poem, they proposed multiple thematic directions, scored them quantitatively, researched Shakespeare's actual sonnets for evidence, built a sourced style guide, and synthesized everything into a final generation.

```
one-shot ──▶ critique: ──▶ propose 3         ──▶ score each route
sonnet       what's        directional             (originality, fit,
             missing?      routes                   resonance, etc.)
                                                        │
                                                        ▼
                                                   recommend hybrid
                                                   of top routes
                                                        │
                    ┌────────────────────────────────────┘
                    ▼
             web research:           web research:
             Shakespeare's     ──▶   style guide
             actual sonnets          (diction, meter,
             (literary review)       rhetoric, imagery)
                    │                      │
                    ▼                      ▼
                    └──── synthesize ──────┘
                              │
                              ▼
                     reformat / validate
                              │
                              ▼
                        final sonnet ✦
```

**What makes this chain interesting:** Most groups iterated on a single poem. This group treated the problem like a research project: diagnose the failure, generate *multiple hypotheses* for how to fix it, score them quantitatively, then research the evidence before writing. The quantified comparison (scoring routes on originality, fit, resonance, conceit strength, difficulty, and authenticity) is a technique borrowed from design thinking. The group also built a full style guide with citations — turning the model into a literary scholar before making it a poet.

---
**Run the cells in order.** Each step builds on the previous one.

## Setup
Run the two cells below once at the start of your session.

In [ ]:
# Install the OpenAI SDK (run once per session)
%pip install openai --quiet
print("✓ Installed")

In [ ]:
from openai import OpenAI
import json
import os

# ── API Key ──────────────────────────────────────────────────────────────────
# In Google Colab:
#   1. Click the 🔑 (Secrets) icon in the left sidebar
#   2. Add a secret named  OPENAI_API_KEY  with your key
#   3. Toggle "Notebook access" to ON, then run this cell
#
# Locally: set the OPENAI_API_KEY environment variable

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
    print("✓ Using Colab Secrets")
except (ImportError, Exception):
    api_key = os.environ.get('OPENAI_API_KEY')
    print("✓ Using environment variable")

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o"

# Helper: call the model and return the text
def ask(prompt, system=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.responses.create(model=MODEL, input=messages)
    return response.output_text

print(f"✓ Client ready | Model: {MODEL}")

---
## Step 1: One-Shot + Critique

Start with the simplest possible prompt, then immediately critique the result. The group found their first draft was "technically correct" but "thematically flat" — the model produced a competent Shakespearean sonnet about time, but with no original conceit and an unearned volta.

In [ ]:
# ── Step 1a: One-shot ────────────────────────────────────────────────────────

first_draft = ask(
    "Write a Shakespearean sonnet."
)

print("FIRST DRAFT")
print("═" * 60)
print(first_draft)

In [ ]:
# ── Step 1b: Critique ────────────────────────────────────────────────────────

critique = ask(
    f"""Here is a Shakespearean sonnet:

{first_draft}

Critique this sonnet. Be specific about what's MISSING, not just
what's wrong:

1. Does it have a genuine central conceit — a metaphor that drives
   the whole poem — or is it just a theme? (Shakespeare's sonnets
   are built on conceits, not topics.)
2. Is the volta (the turn at line 9 or the couplet) earned by the
   preceding argument, or does it just... happen?
3. Is the core idea original, or is it borrowed? ("Time devours all"
   is Shakespeare's idea, not a new one.)
4. Does the poem do anything surprising — something only THIS poem
   could do?

Be direct. The goal is to identify what a second draft needs."""
)

print("CRITIQUE")
print("═" * 60)
print(critique)

---
## Step 2: Propose Three Directional Routes

Instead of just asking the model to "try again," the group asked it to propose **three distinct thematic directions** — each with a different central conceit. This is a design-thinking move: generate multiple options before committing to one.

The group's three routes were:
- **Route 1: Self-Erasure** — writing destroys the self it preserves
- **Route 2: Unreliable Beloved** — the poem works, but for the wrong reader
- **Route 3: Language as Betrayal** — words kill the feeling they try to name

In [ ]:
# ── Step 2: Three directional routes ─────────────────────────────────────────

routes = ask(
    f"""Here is a Shakespearean sonnet and a critique of it:

SONNET:
{first_draft}

CRITIQUE:
{critique}

Based on this critique, propose THREE distinct directional routes
for a new sonnet. Each route should:

- Have a specific, original CONCEIT (not just a theme)
- Be arguable — the conceit should create tension or paradox
- Be achievable in 14 lines of iambic pentameter
- Be different enough from the others that choosing between them
  is a real decision

For each route, give:
1. A name (2–3 words)
2. The central conceit in one sentence
3. How the argument would unfold across three quatrains + couplet
4. What makes this conceit original (not a rehash of Shakespeare's
   own ideas)

Label them Route 1, Route 2, Route 3."""
)

print("THREE DIRECTIONAL ROUTES")
print("═" * 60)
print(routes)

---
## Step 3: Quantified Comparison

Now score each route on specific criteria. The group used six dimensions: **originality, fit, resonance, conceit strength, difficulty, and authenticity**. This forces the model (and you) to think about *why* one direction is better, not just which one "feels right."

The group found that a **hybrid** of Routes 1 and 2 scored higher than any single route.

In [ ]:
# ── Step 3: Quantified comparison ────────────────────────────────────────────

scoring = ask(
    f"""Here are three proposed routes for a Shakespearean sonnet:

{routes}

Score each route on these six criteria (1–10 scale):

1. **Originality** — How fresh is the conceit? Does it feel new?
2. **Fit** — How well does it work in Shakespearean sonnet form?
   (ABAB CDCD EFEF GG, iambic pentameter, volta)
3. **Resonance** — Does the conceit open up meaning, or close it down?
4. **Conceit Strength** — Can the metaphor sustain 14 lines without
   becoming strained?
5. **Difficulty** — How hard will this be to execute well? (Higher = harder
   = more ambitious)
6. **Authenticity** — Would this feel at home in Shakespeare's sequence,
   or does it feel anachronistic?

Present as a table. After scoring, make a recommendation:
- Which single route scores highest?
- Could any two routes be combined into a hybrid that scores even higher?
- If yes, describe the hybrid conceit."""
)

print("QUANTIFIED COMPARISON")
print("═" * 60)
print(scoring)

---
## Step 4: Literary Review — Research Shakespeare's Actual Sonnets

Before writing, the group did **exhaustive research** into Shakespeare's actual sonnets relevant to the chosen routes. This grounds the generation in real textual evidence rather than the model's general knowledge.

The group found specific sonnet clusters:
- **Self-erasure**: Sonnets 71, 72, 73, 74, 62 ("consumed with that which it was nourished by")
- **Unreliable beloved**: Sonnets 17, 18, 32, 55, 81 ("eyes not yet created shall o'er-read")

In [ ]:
# ── Step 4: Literary review ──────────────────────────────────────────────────

literary_review = ask(
    f"""Based on the scoring and recommendation:

{scoring}

Now conduct a literary review of Shakespeare's actual sonnets that
are relevant to the recommended route(s). For each relevant sonnet:

1. Identify the sonnet number and its central conceit
2. Quote the key lines that relate to our chosen direction
3. Analyze how Shakespeare handles the theme — what moves does he
   make that we should learn from?
4. Note any patterns across the group of sonnets (recurring images,
   rhetorical strategies, structural choices)

Look especially for:
- How Shakespeare handles self-reference and meta-poetic themes
- How he addresses the beloved in unexpected ways
- Where his conceits are most original vs. most conventional
- The specific techniques he uses at the volta and couplet

End with a synthesis: what does this research tell us about how to
write our sonnet? Where should we converge with Shakespeare's
approach, and where should we diverge?""",
    system="You are a Shakespeare scholar. Be specific and quote the texts."
)

print("LITERARY REVIEW")
print("═" * 60)
print(literary_review)

---
## Step 5: Style Guide

The group built a comprehensive style guide — not just "how to write a sonnet" but specifically **how Shakespeare writes sonnets**, with sourced rules covering structure, tone, voice, vocabulary, imagery, rhetorical devices, and "anti-rules" (things Shakespeare does that break expected patterns).

This step turns the model into a literary scholar before asking it to be a poet.

In [ ]:
# ── Step 5: Style guide ──────────────────────────────────────────────────────

style_guide = ask(
    """Create a comprehensive Shakespeare Sonnet Style Guide. This should
be a reference document for writing a sonnet that sounds authentically
Shakespearean — not just formally correct, but stylistically convincing.

Organize it into these sections:

1. **Structure** — Beyond ABAB CDCD EFEF GG. How does Shakespeare
   actually use the three quatrains? What kinds of arguments does
   he build? How do his voltas work?

2. **Tone & Voice** — The specific register of Shakespeare's sonnets.
   Where is he earnest vs. ironic? How does he handle direct address?

3. **Vocabulary** — His characteristic diction. Multi-meaning words,
   Latinate vs. Anglo-Saxon choices, legal/financial metaphors,
   the specific words he overuses.

4. **Imagery** — His image patterns: nature, seasons, commerce,
   the body, time. How he extends images across multiple lines.

5. **Rhetorical Devices** — Anaphora, antithesis, chiasmus,
   paradox, syllogism. Which does he use most? How?

6. **Meter** — Iambic pentameter, yes, but where does he break it?
   How does he use feminine endings, trochaic substitutions,
   caesura? When does the meter fight the syntax?

7. **Anti-Rules** — Things Shakespeare does that surprise: breaking
   his own patterns, subverting the couplet, contradicting the
   argument, leaving paradoxes unresolved.

For each rule or pattern, cite specific sonnets as examples.
Aim for 15–20 actionable rules total.""",
    system="You are a Shakespeare scholar and prosodist. Be specific, cite sonnet numbers, and quote lines."
)

print("SHAKESPEARE SONNET STYLE GUIDE")
print("═" * 60)
print(style_guide)

---
## Step 6: Synthesis → Final Sonnet

Now everything converges: the chosen route (hybrid conceit), the literary review (evidence from Shakespeare's actual sonnets), and the style guide (rules for authentic execution). This is the generation step — and it has more context than any other notebook's final generation.

In [ ]:
# ── Step 6: Synthesis ────────────────────────────────────────────────────────

final_sonnet = ask(
    f"""Write a Shakespearean sonnet. You have extensive research to draw on.

CHOSEN DIRECTION (from quantified comparison):
{scoring}

LITERARY REVIEW (Shakespeare's relevant sonnets):
{literary_review}

STYLE GUIDE (rules for authentic execution):
{style_guide}

Requirements:
- Use the recommended hybrid conceit from the scoring step
- Follow the Shakespearean sonnet form exactly: ABAB CDCD EFEF GG,
  iambic pentameter
- Apply the style guide: multi-meaning words, earned volta,
  paradox, strategic metrical breaks
- Draw on the literary review: converge with Shakespeare's actual
  techniques where appropriate, diverge where the conceit demands it
- The conceit should be sustained across all 14 lines, not just
  stated and abandoned
- The couplet should do something unexpected — not just summarize

Write only the sonnet. No title."""
)

print("SYNTHESIZED SONNET")
print("═" * 60)
print(final_sonnet)

---
## Step 7: Reformat + Validate

The final step: validate the sonnet against all the research. Does it actually follow the style guide? Does the conceit hold? Is the meter correct? This is quality control — catching anything the generation step missed.

In [ ]:
# ── Step 7a: Validate ────────────────────────────────────────────────────────

validation = ask(
    f"""Here is a Shakespearean sonnet:

{final_sonnet}

Validate it against these criteria:

1. **Form**: Is it ABAB CDCD EFEF GG? Is the meter iambic pentameter
   throughout? Mark any lines that deviate.

2. **Conceit**: Does the central conceit sustain across all 14 lines?
   Where is it strongest? Where does it thin out?

3. **Volta**: Where is the turn? Is it earned by the preceding argument?

4. **Style Guide compliance**: Check against these key rules:
   - Multi-meaning words present?
   - Paradox or unresolved tension?
   - Strategic metrical breaks (not just smooth pentameter)?
   - Couplet does something unexpected?

5. **Authenticity**: Would this pass as plausibly Shakespearean to a
   careful reader? What gives it away as modern/generated?

For any failures, suggest specific fixes (replacement lines or words).""",
    system="You are a prosodist and Shakespeare scholar. Be precise about meter, rhyme, and form."
)

print("VALIDATION")
print("═" * 60)
print(validation)

In [ ]:
# ── Step 7b: Final version ───────────────────────────────────────────────────

final_poem = ask(
    f"""Here is a Shakespearean sonnet:

{final_sonnet}

Here is a validation report:

{validation}

Apply the fixes identified in the validation. Correct any metrical
errors, strengthen the conceit where it thins, and address any
authenticity issues. Keep everything that passed validation.

Write only the final sonnet."""
)

print("FINAL SONNET ✦")
print("═" * 60)
print(final_poem)

---
## Compare All Versions

The progression here is from a flat first draft to a researched, validated final version. The key question is whether all that research actually produced a *better poem* — or just a more technically correct one.

In [ ]:
# ── Side-by-side comparison ──────────────────────────────────────────────────

print("PROGRESSION")
print("\n" + "═" * 60)
print("1. FIRST DRAFT (one-shot, no context)")
print("═" * 60)
print(first_draft)

print("\n" + "═" * 60)
print("2. SYNTHESIZED (after research + style guide)")
print("═" * 60)
print(final_sonnet)

print("\n" + "═" * 60)
print("3. FINAL (after validation + fixes)")
print("═" * 60)
print(final_poem)

print("\n" + "═" * 60)
print("\nFor your essay, consider:")
print("  → Did the quantified scoring actually help choose a direction,")
print("    or was it just a formality?")
print("  → What did the literary review add that the model's training")
print("    data didn't already include?")
print("  → Did the style guide produce a more authentic sonnet, or")
print("    just a more rule-following one?")
print("  → Is the hybrid conceit (two routes merged) stronger than")
print("    either route alone?")
print("  → This chain is research-heavy. Is there a point of diminishing")
print("    returns — where more research stops improving the poem?")

---
## Going Further

This chain is a starting point. Here are ways to extend it for your assignment:

**Try different route combinations.** The group hybridized Routes 1 and 2. What about 1+3 or 2+3? Generate a sonnet for each hybrid and compare.

**Add more research.** The literary review covered ~10 sonnets. What if you fed in 20? The full 154? Does more evidence improve the output, or overwhelm it?

**Challenge the style guide.** Generate a sonnet that deliberately breaks 3–4 of the style guide's rules. Is it worse — or does rule-breaking produce something more alive?

**Try the chain on a different poet.** This research-heavy approach could work for any poet with a large, well-studied body of work. Try it on Dickinson, Donne, or Petrarch.

**Generate love song lyrics.** The Shakespearean conceit tradition is deeply connected to song (the sonnets may have been performed). Try converting the final sonnet into a song with verses, chorus, and bridge.

**Submitting your work:**
- **Lyrics**: Submit an album's worth of songs, with your favorite first
- **Audio**: Take your best lyrics to [Suno](https://suno.com) and generate audio
- **Essay** (500–700 words): Explain your prompt chain, include sample prompts, and reflect on what GPT-4o got right and wrong about your poet